In [9]:
!pip install openpyxl

  Using cached openpyxl-3.1.5-py2.py3-none-any.whl.metadata (2.5 kB)
  Using cached et_xmlfile-2.0.0-py3-none-any.whl.metadata (2.7 kB)
Using cached openpyxl-3.1.5-py2.py3-none-any.whl (250 kB)
Using cached et_xmlfile-2.0.0-py3-none-any.whl (18 kB)

   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   ---------------------------------------- 2/2 [openpyxl]



In [2]:
import pandas as pd
import os

# ── FILE PATHS ────────────────────────────────────────────────────────────────

STATIC_PATH       = r"C:\Users\sarda\Desktop\convertible_arb\data\raw\bond_static\CONV_BOND_DATA.xlsx"
BOND_DYNAMIC_PATH = r"C:\Users\sarda\Desktop\convertible_arb\data\raw\bond_dynamic/"
EQUITY_PATH       = r"C:\Users\sarda\Desktop\convertible_arb\data\raw\stock/"


# ── MAIN FUNCTION ─────────────────────────────────────────────────────────────

def load_bond_data(ticker):
    """
    Given a bond ticker (e.g. 'MTH'), returns 3 dataframes:

    1. static_data  → 1 row from static file for that ticker
    2. bond_dynamic → daily bond prices for that ticker
    3. equity_data  → daily equity data for the underlying stock

    Usage:
        static_data, bond_dynamic, equity_data = load_bond_data('MTH')
    """

    # ── 1. STATIC BOND DATA ───────────────────────────────────────────────────
    static_df   = pd.read_excel(STATIC_PATH, sheet_name="Sheet1")
    static_data = static_df[static_df['Ticker'] == ticker]

    if static_data.empty:
        raise ValueError(f"Ticker '{ticker}' not found in static bond data.")

    print(f"✅ Static data loaded  — {len(static_data)} row for {ticker}")


    # ── 2. DYNAMIC BOND DATA ──────────────────────────────────────────────────
    bond_file = os.path.join(BOND_DYNAMIC_PATH, f'{ticker}_bond.xlsx')

    if not os.path.exists(bond_file):
        raise FileNotFoundError(f"Bond dynamic file not found: {bond_file}")

    bond_dynamic         = pd.read_excel(bond_file, skiprows=5)
    bond_dynamic.columns = ['date', 'clean_mid', 'dirty_mid']
    bond_dynamic['date'] = pd.to_datetime(bond_dynamic['date'])
    bond_dynamic         = bond_dynamic.sort_values('date').reset_index(drop=True)

    print(f"✅ Bond dynamic loaded — {len(bond_dynamic)} rows | "
          f"{bond_dynamic['date'].min().date()} to {bond_dynamic['date'].max().date()}")


    # ── 3. EQUITY DATA ────────────────────────────────────────────────────────
    equity_file = os.path.join(EQUITY_PATH, f'{ticker}_equity.xlsx')

    if not os.path.exists(equity_file):
        raise FileNotFoundError(f"Equity file not found: {equity_file}")

    equity_data         = pd.read_excel(equity_file, skiprows=6)
    equity_data         = equity_data.rename(columns={
        'Date'                       : 'date',
        'PX_OFFICIAL_CLOSE'          : 'stock_price',
        '12MTH_IMPVOL_100.0%MNY_DF'  : 'implied_vol'
    })
    equity_data['date'] = pd.to_datetime(equity_data['date'])
    equity_data         = equity_data.sort_values('date').reset_index(drop=True)

    print(f"✅ Equity data loaded  — {len(equity_data)} rows | "
          f"{equity_data['date'].min().date()} to {equity_data['date'].max().date()}")


    return static_data, bond_dynamic, equity_data


# ── RUN AND TEST ──────────────────────────────────────────────────────────────

if __name__ == "__main__":

    # Change ticker here to test any bond
    ticker = 'MTH'

    print(f"\n── Loading data for {ticker} ──────────────────────────────────────")
    static_data, bond_dynamic, equity_data = load_bond_data(ticker)

    print(f"\n── Static Data ────────────────────────────────────────────────────")
    print(static_data.T)        # transposed so all columns are visible

    print(f"\n── Bond Dynamic (first 5 rows) ────────────────────────────────────")
    print(bond_dynamic.head())

    print(f"\n── Equity Data (first 5 rows) ─────────────────────────────────────")
    print(equity_data.head())


── Loading data for MTH ──────────────────────────────────────
✅ Static data loaded  — 1 row for MTH
✅ Bond dynamic loaded — 377 rows | 2024-05-07 to 2026-03-04
✅ Equity data loaded  — 455 rows | 2024-05-09 to 2026-03-04

── Static Data ────────────────────────────────────────────────────
                                         13
Issuer Name             Meritage Homes Corp
Ticker                                  MTH
Day Count                            30/360
First Cpn Dt                     11/15/2024
Interest Accrued                    5395.83
Cpn Freq Des                            S/A
CUSIP                             59001ABF8
ISIN                           US59001ABF84
Par Amt                                1000
Volume                                15050
Cmn Stk Tkr                             MTH
Amt Out                           575000000
CV Model Delta S %                25.762157
CV Equity Option Value             3.469858
Conversion Ratio                       8.63
CV Co